In [39]:
# imports
import pandas as pd
import numpy as np
import requests
import time
from bs4 import BeautifulSoup
import re
from selenium import webdriver

In [48]:
# Új dataframe létrehozása
if input("Ha tényleg új dataframe-t szeretnél, írd le hogy 'Igen'") == "Igen":
    cities = {
        "Barcelona": {"country": "Spain"}, 
        "Lisbon": {"country": "Portugal"}, 
        "Tirana": {"country": "Albania"}, 
        "Geneva": {"country": "Switzerland"},
    }

    df = pd.DataFrame(cities).T
    print("Új dataframe:")
    display(df.head(5))
else:
    print("Új dataframe készítése megszakítva.")

Új dataframe:


,country
Barcelona,Spain
Lisbon,Portugal
Tirana,Albania
Geneva,Switzerland


In [36]:
# Földrajz API

# Fő földrajzi típusok Overpass kulcsszavai
geo_types = {
    "beach": "natural=beach",
    "mountain": "natural=peak",
    "lake": "natural=lake",
    "desert": "natural=desert",
    "island": "place=island",
    "attraction": "tourism=attraction",
    "park": "leisure=park", 
    "monument": "historic=monument",
}

def get_coordinates(city, country):
    """
    Visszaadja a (lat, lon) koordinátákat egy város és ország alapján
    """
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "city": city,
        "country": country,
        "format": "json",
        "limit": 1
    }
    
    response = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    
    if response.status_code == 200 and response.json():
        data = response.json()[0]
        lat = float(data["lat"])
        lon = float(data["lon"])
        return lat, lon
    else:
        return None, None
    
def get_geo_scores(lat, lon, radius=10000):
    """
    Lekérdezi az Overpass API-t, és visszaadja a fő földrajzi típusokra
    a találatok számát normalizált 0-1 skálán.
    """
    scores = {}
    
    for typ, tag in geo_types.items():
        query = f"""
        [out:json][timeout:25];
        (
          node["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          way["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
          relation["{tag.split('=')[0]}"="{tag.split('=')[1]}"](around:{radius},{lat},{lon});
        );
        out center;
        """
        url = "http://overpass-api.de/api/interpreter"
        response = requests.post(url, data={"data": query})

        time.sleep(2)  # rate limit elkerülése
        
        if response.status_code == 200:
            data = response.json()
            count = len(data["elements"])
            scores[typ] = count
        else:
            scores[typ] = -1  # hiba esetén -1
    
    return scores

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row["country"])
    scores = get_geo_scores(lat, lon)
    print(f"\n{city} földrajzi típus pontszámok:")
    for k, v in scores.items():
        df.loc[city, f"geo_{k}"] = v
        print(f"  {k}: {v:.2f}")



Barcelona földrajzi típus pontszámok:
  beach: 23.00
  mountain: 58.00
  lake: 0.00
  desert: 0.00
  island: -1.00
  attraction: 126.00
  park: 1003.00
  monument: 115.00

Lisbon földrajzi típus pontszámok:
  beach: 30.00
  mountain: 6.00
  lake: 0.00
  desert: 0.00
  island: 0.00
  attraction: 247.00
  park: 669.00
  monument: 27.00

Tirana földrajzi típus pontszámok:
  beach: 0.00
  mountain: 160.00
  lake: 0.00
  desert: 0.00
  island: -1.00
  attraction: -1.00
  park: 255.00
  monument: 16.00

Geneva földrajzi típus pontszámok:
  beach: 42.00
  mountain: 3.00
  lake: 0.00
  desert: -1.00
  island: 0.00
  attraction: 74.00
  park: 1002.00
  monument: 11.00


In [38]:
# Költségek (numbeo)
def clean_col_name(name):
    """
    Tisztítja az oszlopneveket:
    - kisbetűs
    - minden speciális karaktert aláhúzásra cserél
    - többszörös aláhúzásból egyet csinál
    """
    name = name.lower()
    # Cseréljük a nem alfanumerikus karaktereket aláhúzásra
    name = re.sub(r'[^a-z0-9]+', '_', name)
    # Többszörös aláhúzás → 1 aláhúzás
    name = re.sub(r'_+', '_', name)
    # Elejéről és végéről aláhúzás eltávolítása
    name = name.strip('_')
    return name


for city, row in df.iterrows():
    # URL encode a városnévhez, ha van szóköz
    city_url = city.replace(" ", "-")
    url = f"https://www.numbeo.com/cost-of-living/in/{city_url}?displayCurrency=EUR"

    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", {"class": "data_wide_table"})
    if table is None:
        print("  Nincs adat a városhoz.")
        continue

    rows = table.find_all("tr")
    for tr in rows:
        cols = tr.find_all("td")
        if len(cols) >= 2:
            item = cols[0].text.strip()
            value = cols[1].text.strip()
            try:
                value_num = float(value.replace("€","").replace(",","").strip())
            except:
                value_num = None

            # Tisztított oszlopnév
            col_name = "col_" + clean_col_name(item)
            df.loc[city, col_name] = value_num


In [50]:
# Klíma

def get_hourly_weather(latitude, longitude, start_date, end_date):
    # pip install openmeteo-requests
    # pip install requests_cache
    # pip install retry-requests
    
    import openmeteo_requests
    import requests_cache
    import pandas as pd
    from retry_requests import retry
    
    # Open-Meteo API kliens beállítása gyorsítótárral és hibakezeléssel
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # API hívás paramétereinek beállítása
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": latitude,  # Budapest szélességi fok
        "longitude": longitude,  # Budapest hosszúsági fok
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,wind_speed_10m,weathercode"
    }
    
    # API hívás végrehajtása
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    
    # Óránkénti adatok feldolgozása
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(1).ValuesAsNumpy()
    hourly_weathercode = hourly.Variables(2).ValuesAsNumpy()
    
    # Időbélyegek létrehozása
    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=False),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=False),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "temperature_2m": hourly_temperature_2m,
        "wind_speed_10m": hourly_wind_speed_10m,
        "weathercode": hourly_weathercode
    }
    
    # DataFrame létrehozása
    hourly_dataframe = pd.DataFrame(data=hourly_data)
    
    # Időjárás kódok értelmezése
    weather_descriptions = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Intense drizzle",
        56: "Light freezing drizzle",
        57: "Intense freezing drizzle",
        61: "Light rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Light snow",
        73: "Moderate snow",
        75: "Heavy snow",
        77: "Hail",
        80: "Light showers",
        81: "Moderate showers",
        82: "Heavy showers",
        85: "Light snow showers",
        86: "Heavy snow showers",
        95: "Light or moderate thunderstorm",
        96: "Light thunderstorm with hail",
        99: "Severe thunderstorm with hail"
    }
    
    # Az időjárás kódok leírásának hozzáadása a DataFrame-hez
    hourly_dataframe["weather_description"] = hourly_dataframe["weathercode"].map(weather_descriptions)
    # Átváltás m/s-ról km/h-ra
    hourly_dataframe["wind_speed_kmh"] = hourly_dataframe["wind_speed_10m"] * 3.6
    
    # Eredmény kiíratása
    output_df = hourly_dataframe[["date", "temperature_2m", "wind_speed_kmh", "weather_description", "weathercode"]]
    output_df = output_df.copy()
    output_df.rename(columns={"temperature_2m": "temp_celsius"}, inplace=True)
        
    return output_df

for city, row in df.iterrows():
    lat, lon = get_coordinates(city, row['country'])
    if lat is None:
        continue

    # archive API (példa 2023-as év)
    climate_df = get_hourly_weather(lat, lon, "2023-01-01", "2023-12-31")

    # Egyszerű havi aggregáció (átlag hőmérséklet)
    climate_df['month'] = climate_df['date'].dt.month
    monthly_avg = climate_df.groupby('month')['temp_celsius'].mean()

    for month, value in monthly_avg.items():
        col_name = f"climate_temp_mean_{month}"
        df.loc[i, col_name] = value

print(df)


               country  climate_temp_mean_1  climate_temp_mean_2  \
Barcelona        Spain             3.000288             4.255777   
Lisbon        Portugal                  NaN                  NaN   
Tirana         Albania                  NaN                  NaN   
Geneva     Switzerland                  NaN                  NaN   

           climate_temp_mean_3  climate_temp_mean_4  climate_temp_mean_5  \
Barcelona             7.547465             9.190375             14.59901   
Lisbon                     NaN                  NaN                  NaN   
Tirana                     NaN                  NaN                  NaN   
Geneva                     NaN                  NaN                  NaN   

           climate_temp_mean_6  climate_temp_mean_7  climate_temp_mean_8  \
Barcelona            20.953777             22.30143             22.89518   
Lisbon                     NaN                  NaN                  NaN   
Tirana                     NaN                  Na

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time

# Chrome headless mód
chrome_options = Options()
#chrome_options.add_argument("--headless")

# driver path változtasd meg a saját géped szerint
driver = webdriver.Chrome(options=chrome_options)

url = "https://www.tripadvisor.com/Attractions-g187497-Activities-Barcelona_Catalonia.html"
driver.get(url)

# Várunk, amíg betöltődik az oldal
time.sleep(5)  # egyszerű delay, Selenium WebDriverWait is jobb

# Látnivalók kiválasztása
titles = driver.find_elements(By.CSS_SELECTOR, "div.ZvrsW.N.G")

print("Top 10 látnivaló Barcelonában:")
for t in titles[:10]:
    print("Látnivaló:", t.text.strip())

driver.quit()


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# BUD (47.4369, 19.2556), LIS (38.7742, -9.1342)
dist = haversine(47.4369, 19.2556, 38.7742, -9.1342)
print("BUD-LIS távolság:", round(dist, 1), "km")


In [ ]:
import requests

url = "https://www.unwto.org/tourism-statistics"
response = requests.get(url)

print("UNWTO oldal elérhető:", response.status_code)
# Itt manuálisan kell a letöltött CSV-t feldolgozni, mivel dinamikus.


In [ ]:
# Dummy városadatbázis (0-1 normalizált értékek)

# Attribútumok számításának forrása:
## Földrajz:
## Ár: 
## Klíma:
## Életstílus:
## Távolság:
## Zsúfoltság: 

cities = {
    "Lisbon": {
        "földrajz": {"tengerpart": 0.9, "hegy": 0.2, "város": 0.7, "sziget": 0.5, "tópart": 0.2, "sivatag": 0.1},
        "ár": 0.9,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.4, "relax": 1.0, "aktív": 0.5, "kulturális": 1.0, "családbarát": 0.6},
        "távolság": 1.0,
        "zsúfoltság": 0.5,
    },
    "Barcelona": {
        "földrajz": {"tengerpart": 1.0, "hegy": 0.1, "város": 0.9, "sziget": 0.3, "tópart": 0.2, "sivatag": 0.0},
        "ár": 0.5,
        "klíma": 0.9,
        "életstílus": {"bulis": 0.6, "relax": 0.5, "aktív": 0.3, "kulturális": 1.0, "családbarát": 0.5},
        "távolság": 0.5,
        "zsúfoltság": 1.0,
    },
    "Tirana": {
        "földrajz": {"tengerpart": 0.4, "hegy": 0.7, "város": 0.6, "sziget": 0.2, "tópart": 0.3, "sivatag": 0.0},
        "ár": 0.9,
        "klíma": 0.8,
        "életstílus": {"bulis": 0.3, "relax": 0.5, "aktív": 0.6, "kulturális": 1.0, "családbarát": 0.4},
        "távolság": 1.0,
        "zsúfoltság": 0.2,
    }
}

# Dummy user input (csúszkák, multi-select)
user_preferences = {
    "súlyok": {  # nulladik kérdés: mennyire fontos az egyes dimenzió
        "földrajz": 8,
        "ár": 9,
        "klíma": 7,
        "életstílus": 10,
        "távolság": 6,
        "zsúfoltság": 5,
    },
    "földrajz": {"tengerpart": 8, "hegy": 3, "város":5, "sziget":7, "tópart":2, "sivatag":1},
    "ár": 0.9,         
    "klíma": 0.8,      
    "életstílus": {"bulis":4, "relax":8, "aktív":6, "kulturális":7, "családbarát":5},
    "távolság": 0.9,   
    "zsúfoltság": 0.7,
}

In [ ]:
# Normalizált súlyok (összeg=1)
total_weight = sum(user_preferences["súlyok"].values())
weights = {k: v/total_weight for k, v in user_preferences["súlyok"].items()}

# Simple similarity function
def similarity(user_val, city_val):
    return 1 - abs(user_val - city_val)

# Weighted similarity per city
def city_score(user_pref, city_data, weights):
    total = 0
    for attr, weight in weights.items():
        if attr in ["földrajz", "életstílus"]:
            # átlag a multi-select értékekből
            user_vals = user_pref[attr]
            city_vals = city_data[attr]
            sim_vals = []
            for k in user_vals.keys():
                sim_vals.append(similarity(user_vals[k]/10, city_vals[k]))
            avg_sim = sum(sim_vals)/len(sim_vals)
            total += weight * avg_sim
        else:
            # EGYÉNI érték
            user_val = user_pref[attr]
            city_val = city_data[attr]
            sim = similarity(user_val, city_val)
            total += weight * sim
    return total

In [ ]:
# Calculate scores for all cities
city_scores = {}
for city_name, city_data in cities.items():
    score = city_score(user_preferences, city_data, weights)
    city_scores[city_name] = score

# Sort top cities
top_cities = sorted(city_scores.items(), key=lambda x: x[1], reverse=True)

# Output
print("Top ajánlott városok a preferenciáid alapján:")
for city, score in top_cities:
    print(f"{city}: {score:.3f}")